In [ ]:
# ===================== INSTALLS =====================
!pip -q install python-louvain imageio

# ===================== IMPORTS =====================
from google.colab import drive
drive.mount('/content/drive')
import os, ast, pickle, imageio, time
import pandas as pd
import numpy as np
import networkx as nx
from itertools import combinations
from collections import Counter
import matplotlib.pyplot as plt
from scipy.stats import linregress
from networkx.algorithms.community import louvain_communities

# ===================== CONFIG =====================
FILE_PATH   = "/content/bananika.csv"
OUTPUT_BASE = "/content/drive/MyDrive/temporal_star_results4"
os.makedirs(OUTPUT_BASE, exist_ok=True)

TOP_K = 15
WINDOW = 1  # yearly windows
RANDOM_SEED = 0

# ===================== LOAD DATA =====================
df = pd.read_csv(FILE_PATH, engine="python", encoding="latin-1",on_bad_lines="skip")
df = df.loc[:, ~df.columns.str.match(r"^Unnamed")].copy()
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

# ===================== AUTHOR PARSER =====================
def parse_authors(cell):
    if pd.isna(cell): return []
    s = str(cell).strip()
    if s.startswith("[") and s.endswith("]"):
        try:
            return [a.strip() for a in ast.literal_eval(s)]
        except:
            return []
    s = s.replace(" and ", ", ").replace(";", ",")
    return list({a.strip() for a in s.split(",") if a.strip()})

# ===================== BUILD YEARLY GRAPHS =====================
def build_graph(frame):
    edge_w = Counter()
    for _, row in frame.iterrows():
        authors = parse_authors(row["authors"])
        if len(authors) < 2: continue
        for u,v in combinations(sorted(set(authors)),2):
            edge_w[(u,v)] += 1.0
    G = nx.Graph()
    for (u,v), w in edge_w.items():
        G.add_edge(u,v,weight=w)
    return G

years = sorted(df["year"].unique())
year_graphs = {}
for y in years:
    G = build_graph(df[df["year"]==y])
    year_graphs[y] = G

print("Built yearly graphs:", len(year_graphs))

# ===================== CENTRALITIES =====================
def compute_metrics(G):
    if G.number_of_nodes()==0:
        return {}
    deg = nx.degree_centrality(G)
    btw = nx.betweenness_centrality(G, normalized=True)
    clo = nx.closeness_centrality(G)
    try:
        eig = nx.eigenvector_centrality(G, max_iter=1000)
    except:
        eig = nx.pagerank(G)
    return {
        "degree":deg,
        "betweenness":btw,
        "closeness":clo,
        "eigenvector":eig
    }

records=[]
for y,G in year_graphs.items():
    mets = compute_metrics(G)
    for metric,vals in mets.items():
        for a,v in vals.items():
            records.append({
                "author":a,
                "year":y,
                "metric":metric,
                "value":v
            })

metrics_df = pd.DataFrame(records)
metrics_df.to_csv(os.path.join(OUTPUT_BASE,"all_yearly_metrics.csv"),index=False)

# ===================== STAR ANALYSIS =====================
def star_analysis(df_metric):
    rows=[]
    for author,g in df_metric.groupby("author"):
        if len(g)<5: continue
        g=g.sort_values("year")
        slope,_,r,p,_ = linregress(g["year"],g["value"])
        rows.append({
            "author":author,
            "slope":slope,
            "r":r,
            "p":p,
            "years":len(g)
        })
    return pd.DataFrame(rows).sort_values("slope",ascending=False)

all_star_results={}

for metric in ["degree","betweenness","closeness","eigenvector"]:
    dfm = metrics_df[metrics_df.metric==metric]
    slopes = star_analysis(dfm)
    rising = slopes.head(TOP_K)
    dying  = slopes.tail(TOP_K).sort_values("slope")
    rising.to_csv(os.path.join(OUTPUT_BASE,f"{metric}_top_rising.csv"),index=False)
    dying.to_csv(os.path.join(OUTPUT_BASE,f"{metric}_top_dying.csv"),index=False)
    all_star_results[metric]=(rising,dying)

print("Saved rising/dying star tables.")

# ===================== NETWORK GIF PER YEAR =====================
def make_yearly_gif():
    frames=[]
    for y,G in year_graphs.items():
        if G.number_of_nodes()==0: continue

        pos = nx.spring_layout(G, seed=RANDOM_SEED)

        communities = louvain_communities(G, seed=RANDOM_SEED)

        comm_map = {}
        for i, c in enumerate(communities):
            for n in c:
                comm_map[n] = i

        node_colors = [comm_map.get(n, 0) for n in G.nodes()]

        plt.figure(figsize=(8,8))
        nx.draw_networkx_edges(G, pos, alpha=0.15)
        nx.draw_networkx_nodes(G, pos,
                               node_color=node_colors,
                               cmap="tab20",
                               node_size=30)

        plt.title(f"Collaboration Network {y}")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(f"tmp_{y}.png", dpi=200)
        frames.append(imageio.imread(f"tmp_{y}.png"))
        plt.close()

    imageio.mimsave(os.path.join(OUTPUT_BASE,"yearly_collaboration.gif"),
                    frames,
                    fps=0.5)

make_yearly_gif()

print("GIF saved.")
print("ALL RESULTS SAVED TO:",OUTPUT_BASE)

Mounted at /content/drive
Built yearly graphs: 38


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
filepath="/content/drive/MyDrive/temporal_star_results2"
def make_yearly_gif():
    frames=[]
    for y,G in year_graphs.items():
        if G.number_of_nodes()==0: continue

        pos = nx.spring_layout(G, seed=RANDOM_SEED)

        communities = louvain_communities(G, seed=RANDOM_SEED)

        comm_map = {}
        for i, c in enumerate(communities):
            for n in c:
                comm_map[n] = i

        node_colors = [comm_map.get(n, 0) for n in G.nodes()]

        plt.figure(figsize=(8,8))
        nx.draw_networkx_edges(G, pos, alpha=0.15)
        nx.draw_networkx_nodes(G, pos,
                               node_color=node_colors,
                               cmap="tab20",
                               node_size=30)

        plt.title(f"Collaboration Network {y}")
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(f"tmp_{y}.png", dpi=200)
        frames.append(imageio.imread(f"tmp_{y}.png"))
        plt.close()

    imageio.mimsave(os.path.join(OUTPUT_BASE,"yearly_collaboration.gif"),
                    frames,
                    fps=0.5)

make_yearly_gif()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


NameError: name 'year_graphs' is not defined

In [ ]:
df['year'].unique()

array([1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997,
       1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008,
       2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
       2020, 2021, 2022, 2023])

In [ ]:
df2 = pd.read_csv(FILE_PATH)

ParserError: Error tokenizing data. C error: EOF inside string starting at row 19711

In [ ]:
with open(FILE_PATH, 'r', encoding='latin-1') as f:
    for i, line in enumerate(f):
        if 19700 <= i <= 19730:
            print(i, line)


19700 9540,9540,9540,2019,e6385d39ec9394f2f3a354d9d2b88eec,Oracle-Efficient Algorithms for Online Linear Optimization with Bandit Feedback,"Shinji Ito, Daisuke Hatano, Hanna Sumita, Kei Takemura, Takuro Fukunaga, Naonori Kakimura, Ken-Ichi Kawarabayashi","We propose computationally efficient algorithms for \textit{online linear optimization with bandit feedback}, in which a player chooses an \textit{action vector} from a given (possibly infinite) set $\mathcal{A} \subseteq \mathbb{R}^d$, and then suffers a loss that can be expressed as a linear function in action vectors. Although existing algorithms achieve an optimal regret bound of $\tilde{O}(\sqrt{T})$ for $T$ rounds (ignoring factors of $\mathrm{poly} (d, \log T)$), computationally efficient ways of implementing them have not yet been specified, in particular when $|\mathcal{A}|$ is not bounded by a polynomial size in $d$. A standard way to pursue computational efficiency is to assume that we have an efficient algorithm referred t

In [ ]:
df2 = pd.read_csv(
    FILE_PATH,
    engine="python",
    encoding="latin-1",
    on_bad_lines="warn"
)


/tmp/ipython-input-2082531691.py:1: ParserWarning: Skipping line 10667: field larger than field limit (131072)

  df2 = pd.read_csv(
/tmp/ipython-input-2082531691.py:1: ParserWarning: Skipping line 17971: field larger than field limit (131072)

  df2 = pd.read_csv(
/tmp/ipython-input-2082531691.py:1: ParserWarning: Skipping line 19712: unexpected end of data

  df2 = pd.read_csv(


In [ ]:
df2['year'].unique()

array([1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997,
       1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008,
       2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019,
       2020, 2021, 2022, 2023])

In [ ]:
df2.tail(5)

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,year,hash_id,title,authors,abstract,pdf_url,pdf_text,tags_semantic,umap_x,umap_y,dominant_tag,sim_to_attention
19703,19670,19670,19670,2023,d39e3ae9a11b79691709a7a6e06a63d9,Secure Out-of-Distribution Task Generalization...,"Shengzhuang Chen, Long-Kai Huang, Jonathan Ric...",The success of meta-learning on out-of-distrib...,https://proceedings.neurips.cc/paper_files/pap...,secure out-of-distribution task generalization...,"['meta learning', 'multi-task learning', 'tran...",8.831143,3.742682,meta learning,0.735786
19704,19671,19671,19671,2023,d3b93537b521f15613524415dfe43f37,Autodecoding Latent 3D Diffusion Models,"Evangelos Ntavelis, Aliaksandr Siarohin, Kyle ...",Diffusion-based methods have shown impressive ...,https://proceedings.neurips.cc/paper_files/pap...,autodecoding latent 3d diffusion models evange...,"['GANs', 'autoencoders', 'adversarial', 'visio...",12.595321,1.220183,GANs,0.729226
19705,19672,19672,19672,2023,d3e8011c912e651ab2a76e7935a1e464,Physion++: Evaluating Physical Scene Understan...,"Hsiao-Yu Tung, Mingyu Ding, Zhenfang Chen, Dan...",General physical scene understanding requires ...,https://proceedings.neurips.cc/paper_files/pap...,physion++: evaluating physical scene understan...,"['CNNs', 'ViTs', 'autoencoders', 'transfer lea...",7.849056,1.511796,CNNs,0.699465
19706,19673,19673,19673,2023,d40e6e4b3ee6c24f2bf2cb72c2412f4b,HA-ViD: A Human Assembly Video Dataset for Com...,"Hao Zheng, Regina Lee, Yuqian Lu",Understanding comprehensive assembly knowledge...,https://proceedings.neurips.cc/paper_files/pap...,ha-vid: a human assembly video dataset for com...,"['imitation learning', 'multi-task learning', ...",10.995352,0.236543,imitation learning,0.683142
19707,19674,19674,19674,2023,d41b70011dd21ec3de5e019302279551,Classical Simulation of Quantum Circuits: Para...,"Xiao-Yang Liu, Zeliang Zhang",Google's quantum supremacy announcement has r...,https://proceedings.neurips.cc/paper_files/pap...,classical simulation of quantum circuits using...,"['ViTs', 'optimization', 'LLMs', 'RL', 'graphs']",3.491550,10.407701,ViTs,0.734818
